In [1]:
# Importar las librerías necesarias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

In [3]:
# URL del archivo en el bucket público de AWS
url = "https://hybridge-education-machine-learning-datasets.s3.us-east-1.amazonaws.com/Fraud.csv"
df = pd.read_csv(url)
df = df.sample(frac=0.20, random_state=42).reset_index(drop=True)

In [4]:
df.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,278,CASH_IN,330218.42,C632336343,20866.00,351084.42,C834976624,452419.57,122201.15,0,0
1,15,PAYMENT,11647.08,C1264712553,30370.00,18722.92,M215391829,0.00,0.00,0,0
2,10,CASH_IN,152264.21,C1746846248,106589.00,258853.21,C1607284477,201303.01,49038.80,0,0
3,403,TRANSFER,1551760.63,C333676753,0.00,0.00,C1564353608,3198359.45,4750120.08,0,0
4,206,CASH_IN,78172.30,C813403091,2921331.58,2999503.88,C1091768874,415821.90,337649.60,0,0


In [5]:
len(df)

1272524

# ¡¡¡6,362,620 OBSERVACIONES!!!

In [7]:
#Limpieza de datos
df['name_Dest'] = df['nameDest'].str[0]
df = df.drop(['nameDest', 'nameOrig'], axis=1)

In [8]:
df.head()

,step,type,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,name_Dest
0,278,CASH_IN,330218.42,20866.00,351084.42,452419.57,122201.15,0,0,C
1,15,PAYMENT,11647.08,30370.00,18722.92,0.00,0.00,0,0,M
2,10,CASH_IN,152264.21,106589.00,258853.21,201303.01,49038.80,0,0,C
3,403,TRANSFER,1551760.63,0.00,0.00,3198359.45,4750120.08,0,0,C
4,206,CASH_IN,78172.30,2921331.58,2999503.88,415821.90,337649.60,0,0,C


In [9]:
#Codificacion de datos
type_cod = pd.get_dummies(df['type'], prefix='type', drop_first=True, dtype=int)
cod_name_dest = pd.get_dummies(df['name_Dest'], prefix='nameDest', dtype=int)
df = pd.concat([df, type_cod, cod_name_dest], axis=1)

df = df.drop(['type', 'name_Dest'], axis=1)

In [10]:
df.sample(6)

,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER,nameDest_C,nameDest_M
38519,133,46833.37,33859.00,0.00,1888081.92,1934915.28,0,0,1,0,0,0,1,0
209779,249,136386.71,9922.00,146308.71,738936.58,602549.86,0,0,0,0,0,0,1,0
423561,278,4407.04,31114.00,26706.96,0.00,0.00,0,0,0,0,1,0,0,1
723380,526,7626.42,5944.00,0.00,0.00,0.00,0,0,0,0,1,0,0,1
826222,159,105873.61,31073.00,0.00,1866531.25,1972404.86,0,0,1,0,0,0,1,0
952060,379,14887.60,150273.29,135385.68,0.00,0.00,0,0,0,0,1,0,0,1


In [12]:
# Separacion de características y variables objetivo
X = df.drop(columns=["isFraud"])  # Características
y = df["isFraud"]  # Variable del objetivo

# Dividir en conjuntos de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Escalar las características
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Entrenamiento del modelo
model = LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)
model.fit(X_train_scaled, y_train)

# Predicciones
y_pred_train = model.predict(X_train_scaled)
y_pred_test = model.predict(X_test_scaled)

# Evaluación inicial
print("\nReporte de clasificación (Prueba):")
print(classification_report(y_test, y_pred_test))



Reporte de clasificación (Prueba):
              precision    recall  f1-score   support

           0       1.00      0.95      0.97    254193
           1       0.02      0.98      0.05       312

    accuracy                           0.95    254505
   macro avg       0.51      0.97      0.51    254505
weighted avg       1.00      0.95      0.97    254505

